In [2]:
"""
Program 6 — FedNova + Deep Residual Network (ResNet for Tabular)
Dataset : ULB Credit Card Fraud (creditcard.csv)

FedNova fixes the "objective inconsistency" problem in FedAvg:
When banks do different numbers of local steps, FedAvg unfairly
weights their contributions. FedNova normalises each client's
update by its effective local step count, ensuring all banks
contribute proportionally regardless of data size.

ResNet for tabular data uses skip connections to allow very deep
networks without vanishing gradients — significantly outperforms
standard MLPs on structured fraud data.

Full metric suite: AUC, F1, Precision@K, Recall@1%FPR,
KS Statistic, per-bank fairness, σ_AUC, JFI
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS       = 5
FL_ROUNDS     = 60
LOCAL_STEPS   = 50      # FedNova uses step count, not epochs
BATCH_SIZE    = 256
LR            = 0.001
RHO           = 0.9     # FedNova momentum factor
RANDOM_STATE  = 42
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading dataset...")
df     = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
X_raw  = df.drop("Class", axis=1).values.astype(np.float32)
y      = df["Class"].values
scaler = StandardScaler()
X_raw  = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
N_FEAT = X_tr.shape[1]

# ── NON-IID SPLIT ─────────────────────────────────────────────────────────────
def non_iid_split(X, y, n, seed=42):
    rng        = np.random.default_rng(seed)
    fi         = np.where(y == 1)[0]
    li         = np.where(y == 0)[0]
    splits     = np.round(rng.dirichlet(np.ones(n) * 0.4) * len(fi)).astype(int)
    splits[-1] = len(fi) - splits[:-1].sum()
    banks, f0  = [], 0
    lpp        = len(li) // n
    for i in range(n):
        fe  = f0 + splits[i]
        idx = np.concatenate([fi[f0:fe], li[i*lpp:(i+1)*lpp]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        f0  = fe
    return banks

print(f"\nNon-IID split across {N_BANKS} banks:")
bank_data = non_iid_split(X_tr, y_tr, N_BANKS)
for i, (xb, yb) in enumerate(bank_data):
    print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")

# ── DEEP RESIDUAL NETWORK FOR TABULAR DATA ────────────────────────────────────
class ResBlock(nn.Module):
    """Residual block: output = F(x) + x with layer norm."""
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.LayerNorm(dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))

class TabResNet(nn.Module):
    """
    Deep residual network for tabular data.
    Input projection → N residual blocks → classification head.
    Skip connections prevent vanishing gradients across many layers.
    """
    def __init__(self, in_dim, hidden=128, n_blocks=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU()
        )
        self.blocks = nn.Sequential(
            *[ResBlock(hidden, dropout) for _ in range(n_blocks)]
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.head(self.blocks(self.input_proj(x))).squeeze()

def make_loader(X, y, batch_size=BATCH_SIZE):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.float32)),
        batch_size=batch_size, shuffle=True
    )

# ── FEDNOVA LOCAL TRAIN ───────────────────────────────────────────────────────
def local_train_fednova(model, global_state, loader, n_steps=LOCAL_STEPS):
    """
    FedNova local training.
    Returns:
        - updated model
        - normalised gradient (model_diff / effective_steps)
        - effective step count τ_i
    """
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=LR, momentum=RHO,
                          weight_decay=1e-4, nesterov=True)
    criterion = nn.BCELoss()
    device    = next(model.parameters()).device

    # Infinite data loader for exact step count
    data_iter = iter(loader)
    for step in range(n_steps):
        try:
            X_b, y_b = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            X_b, y_b = next(data_iter)

        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # Compute normalised update: Δw_i = (w_local - w_global) / τ_i
    # FedNova aggregates these normalised updates, not raw weights
    tau_i      = n_steps
    norm_grad  = {}
    local_state = model.state_dict()
    for key in local_state:
        if local_state[key].dtype == torch.float32:
            norm_grad[key] = (local_state[key]
                              - global_state[key].to(device)) / tau_i
        else:
            norm_grad[key] = local_state[key].clone()

    return model, norm_grad, tau_i

# ── EVALUATE ─────────────────────────────────────────────────────────────────
def evaluate_full(model, X, y, k_pct=0.005):
    model.eval()
    with torch.no_grad():
        prob = model(torch.tensor(X, dtype=torch.float32).to(
            next(model.parameters()).device)).cpu().numpy()

    pred  = (prob >= 0.5).astype(int)
    auc   = roc_auc_score(y, prob)
    f1    = f1_score(y, pred, zero_division=0)
    prec  = precision_score(y, pred, zero_division=0)
    rec   = recall_score(y, pred, zero_division=0)

    # KS Statistic — key credit scoring metric
    fpr, tpr, _ = roc_curve(y, prob)
    ks   = np.max(tpr - fpr)

    # Recall @ 1% FPR
    idx_1fpr = np.where(fpr <= 0.01)[0]
    r1fpr    = tpr[idx_1fpr[-1]] if len(idx_1fpr) else 0.0

    # Precision @ K
    K  = max(1, int(len(y) * k_pct))
    pk = y[np.argsort(prob)[::-1][:K]].mean()

    return dict(auc=auc, f1=f1, precision=prec, recall=rec,
                ks_stat=ks, recall_1fpr=r1fpr, prec_at_k=pk)

# ── FEDERATED TRAINING ────────────────────────────────────────────────────────
print(f"\nStarting FedNova FL: {FL_ROUNDS} rounds × {LOCAL_STEPS} local steps/bank\n")
global_model = TabResNet(N_FEAT, hidden=128, n_blocks=4).to(DEVICE)

for rnd in range(1, FL_ROUNDS + 1):
    global_state   = copy.deepcopy(global_model.state_dict())
    norm_grads     = []
    tau_list       = []
    local_sizes    = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        try:
            sm      = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b

        local_model = copy.deepcopy(global_model)
        loader      = make_loader(X_r, y_r)

        _, norm_grad, tau_i = local_train_fednova(
            local_model, global_state, loader
        )

        norm_grads.append(norm_grad)
        tau_list.append(tau_i)
        local_sizes.append(len(X_b))

    # ── FedNova aggregation ───────────────────────────────────────────────────
    # w_new = w_global - lr_server * Σ(w_i * τ_i * norm_grad_i) / Σ(w_i * τ_i)
    total_weight = sum(local_sizes[i] * tau_list[i] for i in range(N_BANKS))
    new_state    = copy.deepcopy(global_state)

    for key in new_state:
        if new_state[key].dtype == torch.float32:
            agg = sum(
                norm_grads[i][key].to(DEVICE) * (local_sizes[i] * tau_list[i] / total_weight)
                for i in range(N_BANKS)
            )
            new_state[key] = global_state[key].to(DEVICE) + agg
        else:
            # For non-float keys (e.g. BatchNorm running stats) use simple average
            new_state[key] = sum(
                norm_grads[i][key].to(DEVICE) * (local_sizes[i] / sum(local_sizes))
                for i in range(N_BANKS)
            ).long() if new_state[key].dtype == torch.long else sum(
                norm_grads[i][key].to(DEVICE) * (local_sizes[i] / sum(local_sizes))
                for i in range(N_BANKS)
            )

    global_model.load_state_dict(new_state)

    if rnd % 5 == 0 or rnd == 1:
        m = evaluate_full(global_model, X_te, y_te)
        print(f"  Round {rnd:>3} | AUC: {m['auc']:.4f} | KS: {m['ks_stat']:.4f} "
              f"| Recall@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f}")

# ── FINAL RESULTS ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL FedNova + ResNet RESULTS")
print("="*60)
m = evaluate_full(global_model, X_te, y_te)
for k, v in m.items():
    print(f"  {k:<20}: {v:.4f}")

print("\nPer-Bank AUC + KS (Fairness):")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2: continue
    mb = evaluate_full(global_model, X_b, y_b)
    bank_aucs.append(mb['auc'])
    print(f"  Bank {i+1}: AUC={mb['auc']:.4f} | KS={mb['ks_stat']:.4f} "
          f"| Recall@1%FPR={mb['recall_1fpr']:.4f}")

print(f"\n  σ_AUC              = {np.std(bank_aucs):.4f}")
jfi = sum(bank_aucs)**2 / (len(bank_aucs) * sum(a**2 for a in bank_aucs))
print(f"  Jain Fairness Index = {jfi:.4f}")

# ── FULL COMPARISON TABLE ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("COMPLETE RESULTS ACROSS ALL PROGRAMS (fill in after running all)")
print("="*60)
print(f"{'Paradigm':<28} {'AUC':>6} {'F1':>6} {'R@1%FPR':>8} {'P@K':>6} {'KS':>6} {'σ_AUC':>7}")
print("-"*72)
print(f"{'FedAvg (LR) [Prog 1]':<28} {'0.9557':>6} {'0.7580':>6} {'  —':>8} {'  —':>6} {'  —':>6} {'0.0179':>7}")
print(f"{'FedProx (NN) [Prog 2]':<28} {'0.9746':>6} {'0.7330':>6} {'0.8980':>8} {'  —':>6} {'  —':>6} {'0.0018':>7}")
print(f"{'PersFL+DP (NN) [Prog 3]':<28} {'0.9761':>6} {'0.7589':>6} {'0.9082':>8} {'0.3063':>6} {'  —':>6} {'0.0000':>7}")
print(f"{'FedAvg+XGBoost [Prog 4]':<28} {'TBD':>6} {'TBD':>6} {'TBD':>8} {'TBD':>6} {'  —':>6} {'TBD':>7}")
print(f"{'SCAFFOLD+TabNet [Prog 5]':<28} {'0.9628':>6} {'0.8137':>6} {'0.8776':>8} {'0.2993':>6} {'  —':>6} {'0.0002':>7}")
print(f"{'FedNova+ResNet [Prog 6]':<28} {m['auc']:.4f} {m['f1']:.4f} {m['recall_1fpr']:>8.4f} {m['prec_at_k']:>6.4f} {m['ks_stat']:>6.4f} {np.std(bank_aucs):>7.4f}")

Device: cuda
Loading dataset...

Non-IID split across 5 banks:
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%

Starting FedNova FL: 60 rounds × 50 local steps/bank

  Round   1 | AUC: 0.6538 | KS: 0.2383 | Recall@1%FPR: 0.0510 | P@K: 0.0070
  Round   5 | AUC: 0.7788 | KS: 0.4613 | Recall@1%FPR: 0.2347 | P@K: 0.0599
  Round  10 | AUC: 0.8540 | KS: 0.6315 | Recall@1%FPR: 0.5102 | P@K: 0.1585
  Round  15 | AUC: 0.8874 | KS: 0.7058 | Recall@1%FPR: 0.6939 | P@K: 0.2254
  Round  20 | AUC: 0.9078 | KS: 0.7599 | Recall@1%FPR: 0.7551 | P@K: 0.2535
  Round  25 | AUC: 0.9229 | KS: 0.7730 | Recall@1%FPR: 0.7755 | P@K: 0.2676
  Round  30 | AUC: 0.9349 | KS: 0.7884 | Recall@1%FPR: 0.7959 | P@K: 0.2676
  Round  35 | AUC: 0.9437 | KS: 0.8011 | Recall@1%FPR: 0.8061 | P@K: 0.2746
  Round  40 | AUC: 0.9500 | KS: 0.8259 | Recall@1%FPR: 0.8163 | P@K: 0.2782
  R